# Report dependence: test Sybils with and without report-mates in training (A15b)

`15`'s union split (gas provision tree ∪ bounty report) cut test F1 from about 0.71 to 0.26–0.28. That drop mixes
**report dependence** (test Sybils have no report-mates in training) with a **collapse of training diversity**
(one component supplied 8,260 of the 8,924 training Sybils). This notebook isolates the first.

**Design.** The union components of `15` (copied verbatim, `reporter=False`). Components holding more than 5 % of
all Sybils go into every training set, in both conditions, and never into validation or test. The other components
are split into 5 folds (20 % each, stratified by Sybil, components whole), repeated with fold seeds
`sp.SPLIT_SEEDS[:3]`. For fold k: its gas provision trees are split 50/50 into **k_a** (test) and **k_b**
(stratified, trees whole); fold k+1 is validation; the other 3 folds plus the large components are the base
training set.

- **Held out**: base training only. No report-mate of a k_a Sybil is in training.
- **Seen**: base training plus k_b, minus whole base trees (never from the large components) chosen at random to
  match k_b's rows and Sybils. Same training size; k_a's Sybils now have their k_b report-mates in training.

Test rows, validation rows and training size are the same in both conditions. LightGBM (parameters from `02`,
model seed 42; XGBoost as a replication), training upsampled to 1:1 as in `sp.make_splits`, threshold from
validation. 5 folds × 3 repeats × 2 conditions.

**Prevalence.** The large components are Sybil-dense, so k_a holds about 2 % Sybils, not 4.19 %. Absolute F1 and AP
are not comparable with the main results; **the paired difference (seen − held out) is the result.**

**Results**: `results/22_report_dependence.json`.

## Setup

In [1]:
import os, platform, time, warnings
import numpy as np
import pandas as pd
from scipy import stats
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
import sybil_pipeline as sp
warnings.filterwarnings('ignore')
pd.set_option('display.width', 220)
assert not sp.CODE_COMMIT.endswith('-dirty'), f'Run from a clean working tree (CODE_COMMIT={sp.CODE_COMMIT})'
print('Code commit:', sp.CODE_COMMIT)
PLATFORM = dict(machine=platform.machine(), processor=platform.processor(), system=platform.system(),
                python=platform.python_version(), cpu_count=os.cpu_count(), n_jobs=sp.N_JOBS)
print('Platform:', PLATFORM)
prov = sp.provenance(['15_split_tree_report'])
print(prov.to_string(index=False)); assert prov['dependencies_unchanged'].all()
DATA_DIR = './data'
FEATS = sp.FEATS
XGB_P, LGBM_P = sp.load_selected_params()
df, _, _ = sp.build_master_df(DATA_DIR, verbose=False)
fcfs = pd.read_csv(sp.data_paths(DATA_DIR)['sybil'])
fcfs.columns = fcfs.columns.str.lower().str.strip()
fcfs['address'] = fcfs['address'].str.lower()
assert fcfs['address'].is_unique
REPORT = df['addr'].map(fcfs.set_index('address')['commonwealth report link'])
assert REPORT[df['sybil'] == 1].notna().all() and REPORT[df['sybil'] == 0].isna().all()

Code commit: ef82c78
Platform: {'machine': 'arm64', 'processor': 'arm', 'system': 'Darwin', 'python': '3.13.7', 'cpu_count': 10, 'n_jobs': 4}


            notebook  commit  dependencies_unchanged
15_split_tree_report 3bbec2c                    True


## Union components (as in `15`) and helpers

In [2]:
def union_groups(df, fcfs, reporter=False):
    """Component id per row: rows joined if they share a provision tree or a bounty report.

    reporter=True also joins rows with the same bounty hunter (`reward address1`), the stricter variant.
    """
    F = fcfs.set_index('address')
    rep = df['addr'].map(F['commonwealth report link']).fillna('').astype(str)
    gh = df['addr'].map(F['github issue']).fillna('').astype(str)
    cols = [
        pd.Series('T:' + df['provision_tree'].astype(str), index=df.index),
        pd.Series(np.where(rep != '', 'R:' + rep, None), index=df.index),
        pd.Series(np.where(gh != '', 'G:' + gh, None), index=df.index),
    ]
    if reporter:
        ra = df['addr'].map(F['reward address1']).fillna('').astype(str)
        cols.append(pd.Series(np.where(ra != '', 'A:' + ra, None), index=df.index))
    keys = pd.concat(cols, axis=1)               # one column per grouping; None = no key
    # pandas >= 3 keeps missing values in stack(); dropna() is required, or None keys would
    # factorize to -1 and silently link unrelated rows
    long = keys.stack().dropna().rename('key').reset_index(level=1, drop=True)   # (row, key) pairs
    row_pos = df.index.get_indexer(long.index)
    key_id, key_uni = pd.factorize(long.values)
    assert (key_id >= 0).all() and (row_pos >= 0).all()
    n_rows, n_keys = len(df), len(key_uni)
    # bipartite graph rows <-> keys; components = union groups
    g = coo_matrix((np.ones(len(long)), (row_pos, n_rows + key_id)),
                   shape=(n_rows + n_keys, n_rows + n_keys))
    _, comp = connected_components(g, directed=False)
    return pd.Series(comp[:n_rows], index=df.index).astype(str)


def group_kway(groups, y, fracs, seed):
    """k-way stratified group partition; generalizes sp._group_partition (same algorithm).

    groups: pd.Series of group ids (index = row index); y: pd.Series of 0/1 (same index);
    fracs: target fraction per part (sums to 1). Returns a pd.Series of part numbers 0..k-1.
    Multi-row groups go first, largest first (random order within equal sizes), each into the
    part with the most remaining room relative to target, weighted by the group's class mix;
    single-row groups then fill each part's remaining per-class quota at random.
    """
    fracs = np.asarray(fracs, dtype=float); k = len(fracs)
    rng = np.random.default_rng(seed)
    yv = y.to_numpy().astype(int)
    n_cls = np.array([(yv == 0).sum(), (yv == 1).sum()])
    target = np.outer(fracs, n_cls).astype(float)
    quota = target.copy()
    g = pd.DataFrame({'grp': groups.to_numpy(), 'y': yv}).groupby('grp', sort=False)['y'].agg(['size', 'sum'])
    multi = g[g['size'] > 1]
    multi = multi.iloc[rng.permutation(len(multi))].sort_values('size', ascending=False, kind='stable')
    part_of = {}
    with np.errstate(divide='ignore', invalid='ignore'):
        for grp, (size, n_s) in zip(multi.index, multi[['size', 'sum']].to_numpy()):
            mix = np.array([size - n_s, n_s]) / np.maximum(n_cls, 1)
            room = np.where(target > 0, quota / target, 0.0)
            p = int(np.argmax(room @ mix))
            quota[p] -= (size - n_s, n_s)
            part_of[grp] = p
    part = np.array(groups.map(part_of), dtype=float)   # np.array copies (pandas 3 copy-on-write returns read-only views)
    single = np.isnan(part)
    for c in (0, 1):
        idx = rng.permutation(np.flatnonzero(single & (yv == c)))
        take = np.maximum(np.round(quota[:, c]), 0).astype(int)
        start = 0
        for p in range(k - 1, 0, -1):            # last parts first; the remainder goes to part 0
            part[idx[start:start + take[p]]] = p
            start += take[p]
        part[idx[start:]] = 0
    out = pd.Series(part.astype(int), index=groups.index)
    # every group sits wholly in one part
    assert (pd.DataFrame({'g': groups, 'p': out}).groupby('g')['p'].nunique() == 1).all()
    return out


def matched_tree_removal(pool, n_rows, n_syb, seed):
    """Pick whole trees from `pool` whose total rows and Sybils match (n_rows, n_syb).

    pool: DataFrame indexed by tree id with columns rows, sybils (trees eligible for removal).
    Sybil-containing trees are drawn at random until the Sybil total is closest to n_syb;
    Sybil-free trees are then drawn at random until the row total is closest to n_rows.
    Returns the list of tree ids to remove.
    """
    rng = np.random.default_rng(seed)
    pick, r, s = [], 0, 0
    syb_trees = pool[pool['sybils'] > 0]
    for t in syb_trees.index[rng.permutation(len(syb_trees))]:
        tr_r, tr_s = pool.at[t, 'rows'], pool.at[t, 'sybils']
        if abs(s + tr_s - n_syb) < abs(s - n_syb):        # only if it brings the Sybil total closer
            pick.append(t); r += tr_r; s += tr_s
        if s >= n_syb:
            break
    clean = pool[pool['sybils'] == 0]
    for t in clean.index[rng.permutation(len(clean))]:
        tr_r = pool.at[t, 'rows']
        if abs(r + tr_r - n_rows) < abs(r - n_rows):
            pick.append(t); r += tr_r
        if r >= n_rows:
            break
    return pick


def build_conditions(df, comp, big, k, fold_of, ab_seed, rm_seed):
    """Row indices for fold k under both conditions. Pure index bookkeeping, no model.

    df: needs 'provision_tree', 'sybil' (index = row index); comp: union component per row;
    big: set of large component ids (always in training); fold_of: fold per row (-1 for big rows).
    Returns dict(k_a, k_b, val, base, held_out_train, seen_train, removed_trees).
    """
    nf = int(fold_of.max()) + 1
    in_k = fold_of == k
    val = df.index[fold_of == (k + 1) % nf]
    base = df.index[(fold_of != k) & (fold_of != (k + 1) % nf)]          # 3 folds + big components (fold -1)
    # split fold k's gas provision trees 50/50, stratified by Sybil, trees whole
    fk = df[in_k]
    ab = group_kway(fk['provision_tree'], fk['sybil'], [0.5, 0.5], seed=ab_seed)
    k_a, k_b = fk.index[ab == 0], fk.index[ab == 1]
    # remove whole base trees (outside the big components) matching k_b's rows and Sybils
    elig = df.loc[base][~comp.loc[base].isin(big)]
    pool = elig.groupby('provision_tree')['sybil'].agg(rows='size', sybils='sum')
    rm = matched_tree_removal(pool, len(k_b), int(df.loc[k_b, 'sybil'].sum()), seed=rm_seed)
    rm_idx = elig.index[elig['provision_tree'].isin(rm)]
    seen = base.difference(rm_idx).union(k_b)
    return dict(k_a=k_a, k_b=k_b, val=val, base=base, held_out_train=base, seen_train=seen,
                removed_trees=rm)


from sklearn.utils import resample

def make_S(df, feats, train_idx, val_idx, test_idx, seed):
    """The same dict as sp.make_splits, built from given row indices (same upsampling code)."""
    X = df[feats].astype(np.float32); y = df['sybil'].astype(int)
    trn = pd.concat([X.loc[train_idx], y.loc[train_idx].rename('sybil')], axis=1)
    maj = trn[trn['sybil'] == 0]; mn = trn[trn['sybil'] == 1]
    mn_up = resample(mn, replace=True, n_samples=len(maj), random_state=seed)
    bal = pd.concat([maj, mn_up]).sample(frac=1, random_state=seed)
    return dict(method='report_dependence',
                X_train=bal.drop(columns='sybil').astype(np.float32), y_train=bal['sybil'],
                X_val=X.loc[val_idx], y_val=y.loc[val_idx], X_test=X.loc[test_idx], y_test=y.loc[test_idx],
                idx_train=train_idx, idx_val=val_idx, idx_test=test_idx)

In [3]:
comp = union_groups(df, fcfs)                       # as in 15
syb_by = df.groupby(comp)['sybil'].sum()
rows_by = comp.value_counts()
N_SYB = int(df['sybil'].sum())
big = set(syb_by[syb_by > 0.05 * N_SYB].index)
rest = ~comp.isin(big)
r15 = sp.load_results('15_split_tree_report')['composition']
top = syb_by.idxmax()
print(f"{comp.nunique():,} components; largest {rows_by[top]:,} rows, {syb_by[top]:,} Sybils; "
      f"large: {len(big)} components, {int((~rest).sum()):,} rows, {int(df.loc[~rest, 'sybil'].sum()):,} Sybils")
assert comp.nunique() == 355_619 and rows_by[top] == 13_346 and syb_by[top] == 8_260
assert int((~rest).sum()) == r15['big_rows'] == 14_300 and int(df.loc[~rest, 'sybil'].sum()) == r15['big_sybils'] == 9_214

355,619 components; largest 13,346 rows, 8,260 Sybils; large: 2 components, 14,300 rows, 9,214 Sybils


## Folds, conditions and fits

In [4]:
def reports_of(idx):
    s = df.loc[idx, 'sybil'] == 1
    return set(REPORT.loc[idx][s])

rows, comp_rows, dose_rows, cat_rows = [], [], [], []
FOLDS = {}
t_all = time.time()
for r, rep_seed in enumerate(sp.SPLIT_SEEDS[:3]):
    fold_of = pd.Series(-1, index=df.index)
    fold_of[rest] = group_kway(comp[rest], df['sybil'][rest], [0.2] * 5, seed=rep_seed)
    # fold checks
    n_rest, s_rest = int(rest.sum()), int(df.loc[rest, 'sybil'].sum())
    for k in range(5):
        f = fold_of == k
        assert abs(f.sum() / n_rest - 0.2) <= 0.01 and abs(df.loc[f, 'sybil'].sum() / s_rest - 0.2) <= 0.01
    assert (fold_of[~rest] == -1).all() and (fold_of[rest] >= 0).all()
    assert (pd.DataFrame({'t': df['provision_tree'][rest], 'f': fold_of[rest]}).groupby('t')['f'].nunique() == 1).all()
    rep_f = pd.DataFrame({'rep': REPORT[rest], 'f': fold_of[rest]}).dropna()
    assert (rep_f.groupby('rep')['f'].nunique() == 1).all()
    FOLDS[rep_seed] = fold_of.value_counts().sort_index().to_dict()
    for k in range(5):
        t0 = time.time()
        C = build_conditions(df, comp, big, k, fold_of, ab_seed=1000 * rep_seed + k, rm_seed=2000 * rep_seed + k)
        ka, kb, val = C['k_a'], C['k_b'], C['val']
        H, Se = C['held_out_train'], C['seen_train']
        # acceptance checks for this fold-repeat
        for a, b in [(ka, val), (ka, H), (ka, Se), (val, H), (val, Se)]:
            assert len(a.intersection(b)) == 0
        assert not set(df.loc[ka, 'provision_tree']) & set(df.loc[kb, 'provision_tree'])
        assert not reports_of(ka) & reports_of(H)                       # no report-mate in held-out training
        hs, ss = int(df.loc[H, 'sybil'].sum()), int(df.loc[Se, 'sybil'].sum())
        assert abs(len(Se) - len(H)) <= 0.01 * len(H) and abs(ss - hs) <= 0.01 * hs, (len(H), len(Se), hs, ss)
        # report-mates in k_b for each k_a Sybil
        ka_syb = ka[df.loc[ka, 'sybil'].to_numpy() == 1]
        kb_counts = REPORT.loc[kb][df.loc[kb, 'sybil'] == 1].value_counts()
        mates = REPORT.loc[ka_syb].map(kb_counts).fillna(0).astype(int)
        res, scores = {}, {}
        for cond, tr in [('held_out', H), ('seen', Se)]:
            S = make_S(df, FEATS, tr, val, ka, seed=sp.SEED)
            assert (S['X_test'].index == ka).all() and (S['X_val'].index == val).all()
            fits = {'LightGBM': sp.fit_lgbm(S, LGBM_P, seeds=(42,), verbose=False),
                    'XGBoost': sp.fit_xgb(S, XGB_P, seeds=(42,), verbose=False)}
            for model, fit in fits.items():
                e = sp.evaluate(f'{model} {cond}', S['y_val'], fit['val'], S['y_test'], fit['test'])
                assert e['threshold'] == sp.best_f1_threshold(S['y_val'], fit['val'])   # threshold from validation
                res[(model, cond)] = e; scores[(model, cond)] = fit['test']
                rows.append(dict(repeat=r, fold_seed=rep_seed, k=k, model=model, condition=cond,
                                 **{m: float(e[m]) for m in ['threshold', 'precision', 'recall', 'f1', 'auroc', 'ap']},
                                 rounds=fit['rounds'][0]))
                bc = sp.metrics_by_category(S['y_test'], fit['test'], df.loc[ka, 'category'], e['threshold'])
                cat_rows += [dict(repeat=r, k=k, model=model, condition=cond, **x) for x in bc.to_dict('records')]
            comp_rows.append(dict(repeat=r, k=k, condition=cond, train_unique=len(tr),
                                  train_sybils=int(df.loc[tr, 'sybil'].sum()), train_reports=len(reports_of(tr)),
                                  big_share_of_train_sybils=float(df.loc[tr, 'sybil'][~rest.loc[tr]].sum() / df.loc[tr, 'sybil'].sum())))
        # per-Sybil scores for the dose-response
        pos = np.flatnonzero(df.loc[ka, 'sybil'].to_numpy() == 1)
        for model in ['LightGBM', 'XGBoost']:
            dose_rows += [dict(repeat=r, k=k, model=model, mates=int(m),
                               score_held=float(scores[(model, 'held_out')][i]), score_seen=float(scores[(model, 'seen')][i]),
                               flag_held=bool(scores[(model, 'held_out')][i] >= res[(model, 'held_out')]['threshold']),
                               flag_seen=bool(scores[(model, 'seen')][i] >= res[(model, 'seen')]['threshold']))
                          for i, m in zip(pos, mates.to_numpy())]
        cat_ka = df.loc[ka, 'category'][df.loc[ka, 'sybil'] == 1].value_counts()
        comp_rows.append(dict(repeat=r, k=k, condition='k_a', test_rows=len(ka), test_sybils=len(ka_syb),
                              test_prevalence=len(ka_syb) / len(ka), val_rows=len(val), val_sybils=int(df.loc[val, 'sybil'].sum()),
                              kb_rows=len(kb), kb_sybils=int(df.loc[kb, 'sybil'].sum()), removed_trees=len(C['removed_trees']),
                              mate_coverage=float((mates > 0).mean()),
                              **{f'test_sybils_{c}': int(cat_ka.get(c, 0)) for c in ['IxL', 'IxI', 'IxE', 'No provider']}))
        lg = {c: res[('LightGBM', c)] for c in ['held_out', 'seen']}
        print(f"repeat {r} fold {k}: k_a {len(ka):,} rows / {len(ka_syb):,} Sybils, mate coverage {(mates > 0).mean():.2f}; "
              f"LightGBM F1 held {lg['held_out']['f1']:.4f} seen {lg['seen']['f1']:.4f}, AUROC {lg['held_out']['auroc']:.4f} / "
              f"{lg['seen']['auroc']:.4f} ({time.time() - t0:.0f}s)", flush=True)
per_fold = pd.DataFrame(rows); composition = pd.DataFrame(comp_rows)
dose = pd.DataFrame(dose_rows); by_cat_raw = pd.DataFrame(cat_rows)
print(f'total {time.time() - t_all:.0f}s')

repeat 0 fold 0: k_a 42,050 rows / 901 Sybils, mate coverage 0.98; LightGBM F1 held 0.4644 seen 0.6249, AUROC 0.9221 / 0.9766 (158s)


repeat 0 fold 1: k_a 42,048 rows / 899 Sybils, mate coverage 0.98; LightGBM F1 held 0.4053 seen 0.5877, AUROC 0.9126 / 0.9759 (169s)


repeat 0 fold 2: k_a 42,048 rows / 899 Sybils, mate coverage 0.99; LightGBM F1 held 0.3218 seen 0.5936, AUROC 0.8871 / 0.9579 (116s)


repeat 0 fold 3: k_a 42,049 rows / 900 Sybils, mate coverage 0.99; LightGBM F1 held 0.2070 seen 0.6025, AUROC 0.8139 / 0.9598 (179s)


repeat 0 fold 4: k_a 42,049 rows / 900 Sybils, mate coverage 0.99; LightGBM F1 held 0.4535 seen 0.6257, AUROC 0.9189 / 0.9731 (186s)


...

repeat 2 fold 1: k_a 42,048 rows / 899 Sybils, mate coverage 0.99; LightGBM F1 held 0.3968 seen 0.5697, AUROC 0.9246 / 0.9771 (119s)


repeat 2 fold 2: k_a 42,048 rows / 899 Sybils, mate coverage 0.98; LightGBM F1 held 0.3209 seen 0.6108, AUROC 0.8839 / 0.9683 (95s)


repeat 2 fold 3: k_a 42,048 rows / 899 Sybils, mate coverage 0.98; LightGBM F1 held 0.2092 seen 0.5810, AUROC 0.8012 / 0.9615 (151s)


repeat 2 fold 4: k_a 42,048 rows / 899 Sybils, mate coverage 0.99; LightGBM F1 held 0.4263 seen 0.5949, AUROC 0.9167 / 0.9738 (137s)


total 3170s


## 1. Main result

Paired difference seen − held out over the 15 fold-repeats. Headline test: the corrected repeated k-fold t-test
(Nadeau and Bengio 2003; Bouckaert and Frank 2004), `t = mean(d) / sqrt((1/15 + n_test/n_train) · var(d))`, df = 14,
with `n_test/n_train` the mean of `len(k_a) / unique training rows`. Wilcoxon signed-rank, uncorrected.

In [5]:
kk = composition[composition['condition'] == 'k_a'].set_index(['repeat', 'k'])
tr = composition[composition['condition'] == 'held_out'].set_index(['repeat', 'k'])
RATIO = float((kk['test_rows'] / tr['train_unique']).mean())
N = 15
summary, tests = {}, []
for model in ['LightGBM', 'XGBoost']:
    P = per_fold[per_fold['model'] == model].pivot_table(index=['repeat', 'k'], columns='condition', values=['f1', 'ap', 'auroc', 'recall', 'precision'])
    assert len(P) == N
    summary[model] = {f'{m}_{c}_{a}': float(getattr(P[(m, c)], a)()) for m in ['f1', 'ap', 'auroc', 'recall', 'precision']
                      for c in ['held_out', 'seen'] for a in ('mean', 'std')}
    for m in ['f1', 'ap', 'auroc', 'recall', 'precision']:
        d = (P[(m, 'seen')] - P[(m, 'held_out')]).to_numpy()
        v = d.var(ddof=1)
        t = d.mean() / np.sqrt((1 / N + RATIO) * v) if v > 0 else np.nan
        tests.append(dict(model=model, metric=m, held_out_mean=float(P[(m, 'held_out')].mean()), seen_mean=float(P[(m, 'seen')].mean()),
                          diff_mean=float(d.mean()), diff_sd=float(d.std(ddof=1)), seen_higher_n=int((d > 0).sum()),
                          t_corrected=float(t), p_corrected=float(2 * stats.t.sf(abs(t), N - 1)),
                          p_wilcoxon_uncorrected=float(stats.wilcoxon(d).pvalue) if np.any(d != 0) else np.nan))
tests = pd.DataFrame(tests)
print(f'n_test / n_train = {RATIO:.4f}')
print(tests.round(4).to_string(index=False))

n_test / n_train = 0.1577
   model    metric  held_out_mean  seen_mean  diff_mean  diff_sd  seen_higher_n  t_corrected  p_corrected  p_wilcoxon_uncorrected
LightGBM        f1         0.3667     0.5981     0.2314   0.0843             15       5.7924       0.0000                  0.0001
LightGBM        ap         0.3048     0.5966     0.2918   0.0748             15       8.2341       0.0000                  0.0001
LightGBM     auroc         0.8913     0.9698     0.0785   0.0411             15       4.0329       0.0012                  0.0001
LightGBM    recall         0.4235     0.7233     0.2998   0.1103             15       5.7379       0.0001                  0.0001
LightGBM precision         0.3354     0.5150     0.1796   0.0804             15       4.7145       0.0003                  0.0001
 XGBoost        f1         0.3633     0.5803     0.2170   0.0810             15       5.6570       0.0001                  0.0001
 XGBoost        ap         0.3055     0.5969     0.2914   0.0777

## 2. Dose-response

For each k_a Sybil, its report-mates in k_b (Sybils in k_b from the same Commonwealth report), binned 0, 1–9,
10–99, ≥ 100. Pooled over the 15 fold-repeats: recall at each condition's validation threshold, the mean score, and
the mean paired score difference per Sybil.

In [6]:
BINS = [-1, 0, 9, 99, np.inf]; LAB = ['0', '1–9', '10–99', '≥100']
dose['bin'] = pd.cut(dose['mates'], BINS, labels=LAB).astype(str)
dose_resp = (dose.assign(gain=dose['score_seen'] - dose['score_held'])
             .groupby(['model', 'bin']).agg(sybils=('gain', 'size'), recall_held=('flag_held', 'mean'), recall_seen=('flag_seen', 'mean'),
                                            score_held=('score_held', 'mean'), score_seen=('score_seen', 'mean'), score_gain=('gain', 'mean'),
                                            score_gain_sd=('gain', 'std'))
             .reindex(pd.MultiIndex.from_product([['LightGBM', 'XGBoost'], LAB])).reset_index().rename(columns={'level_0': 'model', 'level_1': 'bin'}))
dose_resp['recall_gain'] = dose_resp['recall_seen'] - dose_resp['recall_held']
print(dose_resp.round(4).to_string(index=False))

   model   bin  sybils  recall_held  recall_seen  score_held  score_seen  score_gain  score_gain_sd  recall_gain
LightGBM     0     181       0.1215       0.1271      0.1214      0.1358      0.0145         0.1215       0.0055
LightGBM   1–9    1275       0.1937       0.4322      0.2041      0.4535      0.2493         0.3325       0.2384
LightGBM 10–99    6111       0.4166       0.7734      0.3990      0.7620      0.3629         0.3838       0.3567
LightGBM  ≥100    5927       0.4893       0.7525      0.4631      0.7376      0.2745         0.3502       0.2632
 XGBoost     0     181       0.1160       0.1215      0.1146      0.1314      0.0167         0.1324       0.0055
 XGBoost   1–9    1275       0.1945       0.4376      0.2039      0.4521      0.2481         0.3251       0.2431
 XGBoost 10–99    6111       0.4165       0.7763      0.4073      0.7646      0.3573         0.3806       0.3598
 XGBoost  ≥100    5927       0.4986       0.7645      0.4806      0.7501      0.2695         0.3

## 3. By category (paired seen − held out, LightGBM and XGBoost)

In [7]:
bc = by_cat_raw[by_cat_raw['category'].isin(['IxL', 'IxI', 'IxE'])]
Pc = bc.pivot_table(index=['model', 'category', 'repeat', 'k'], columns='condition', values=['f1', 'recall', 'auroc'])
by_category = []
for (model, cat), g in Pc.groupby(level=['model', 'category']):
    row = dict(model=model, category=cat, test_sybils_per_fold=float(bc[(bc.model == model) & (bc.category == cat) & (bc.condition == 'seen')]['sybil'].mean()))
    for m in ['f1', 'recall', 'auroc']:
        d = (g[(m, 'seen')] - g[(m, 'held_out')]).to_numpy()
        row.update({f'{m}_held_out': float(g[(m, 'held_out')].mean()), f'{m}_seen': float(g[(m, 'seen')].mean()),
                    f'{m}_diff_mean': float(np.nanmean(d)), f'{m}_diff_sd': float(np.nanstd(d, ddof=1)), f'{m}_seen_higher_n': int((d > 0).sum())})
    by_category.append(row)
by_category = pd.DataFrame(by_category)
print(by_category.round(4).to_string(index=False))

   model category  test_sybils_per_fold  f1_held_out  f1_seen  f1_diff_mean  f1_diff_sd  f1_seen_higher_n  recall_held_out  recall_seen  recall_diff_mean  recall_diff_sd  recall_seen_higher_n  auroc_held_out  auroc_seen  auroc_diff_mean  auroc_diff_sd  auroc_seen_higher_n
LightGBM      IxE               28.1333       0.0680   0.2217        0.1537      0.2363                13           0.0611       0.1784            0.1173          0.1897                     8          0.8543      0.8983           0.0440         0.0356                   13
LightGBM      IxI               11.2667       0.0103   0.0398        0.0296      0.0661                 3           0.0095       0.0346            0.0251          0.0555                     3          0.7976      0.8218           0.0242         0.0465                   12
LightGBM      IxL              859.6667       0.3767   0.6083        0.2317      0.0819                15           0.4396       0.7480            0.3083          0.1120            

## 4. Composition

In [8]:
ka_c = composition[composition['condition'] == 'k_a']
trc = composition[composition['condition'].isin(['held_out', 'seen'])]
print(ka_c[['repeat', 'k', 'test_rows', 'test_sybils', 'test_prevalence', 'val_sybils', 'kb_sybils', 'removed_trees', 'mate_coverage',
            'test_sybils_IxL', 'test_sybils_IxI', 'test_sybils_IxE']].round(4).to_string(index=False))
print(trc.groupby('condition')[['train_unique', 'train_sybils', 'train_reports', 'big_share_of_train_sybils']].agg(['mean', 'min', 'max']).round(4).to_string())
tree_share = r15['by_partition'][0]['sybils_in_BIG'] / r15['by_partition'][0]['sybils']
print(f"Tree split (15, seed 42): {tree_share:.3f} of training Sybils from the large components; union split: "
      f"{r15['by_partition'][3]['sybils_in_BIG'] / r15['by_partition'][3]['sybils']:.3f}")
comp_summary = dict(
    folds_by_seed=FOLDS,
    k_a_sybils_min=int(ka_c['test_sybils'].min()), k_a_sybils_max=int(ka_c['test_sybils'].max()),
    k_a_prevalence_mean=float(ka_c['test_prevalence'].mean()),
    mate_coverage_mean=float(ka_c['mate_coverage'].mean()), mate_coverage_min=float(ka_c['mate_coverage'].min()),
    mate_coverage_max=float(ka_c['mate_coverage'].max()),
    train_sybils_held_mean=float(trc[trc.condition == 'held_out']['train_sybils'].mean()),
    train_sybils_seen_mean=float(trc[trc.condition == 'seen']['train_sybils'].mean()),
    train_reports_held_mean=float(trc[trc.condition == 'held_out']['train_reports'].mean()),
    train_reports_seen_mean=float(trc[trc.condition == 'seen']['train_reports'].mean()),
    big_share_held_mean=float(trc[trc.condition == 'held_out']['big_share_of_train_sybils'].mean()),
    big_share_seen_mean=float(trc[trc.condition == 'seen']['big_share_of_train_sybils'].mean()),
    tree_split_big_share=float(tree_share), n_test_over_n_train=RATIO)
print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in comp_summary.items() if k != 'folds_by_seed'})

 repeat  k  test_rows  test_sybils  test_prevalence  val_sybils  kb_sybils  removed_trees  mate_coverage  test_sybils_IxL  test_sybils_IxI  test_sybils_IxE
      0  0    42050.0        901.0           0.0214      1799.0      900.0        36362.0         0.9834            890.0              6.0              5.0
      0  1    42048.0        899.0           0.0214      1799.0      900.0        36499.0         0.9766            874.0              7.0             18.0
      0  2    42048.0        899.0           0.0214      1799.0      900.0        36576.0         0.9889            836.0              7.0             55.0
      0  3    42049.0        900.0           0.0214      1799.0      899.0        36896.0         0.9911            832.0             15.0             53.0
      0  4    42049.0        900.0           0.0214      1801.0      899.0        36023.0         0.9911            861.0             16.0             22.0
      1  0    42049.0        901.0           0.0214      1799.0 

## Save results

In [9]:
sp.save_results([], '22_report_dependence', extra=dict(
    per_fold=per_fold.to_dict('records'), composition=composition.to_dict('records'), composition_summary=comp_summary,
    summary=summary, tests=tests.to_dict('records'), dose_response=dose_resp.to_dict('records'),
    by_category=by_category.to_dict('records'),
    test_method=dict(headline='corrected repeated k-fold t-test (Nadeau and Bengio 2003; Bouckaert and Frank 2004), df = 14',
                     n_test_over_n_train=RATIO, also='Wilcoxon signed-rank, uncorrected'),
    design=dict(folds=5, repeats=3, fold_seeds=list(sp.SPLIT_SEEDS[:3]), big_components=len(big),
                big_rows=int((~rest).sum()), big_sybils=int(df.loc[~rest, 'sybil'].sum()),
                note='only the paired difference seen - held out is the result; k_a prevalence is about 2%'),
    params=dict(lgbm=LGBM_P, xgb=XGB_P, source='02_hyperparameter_search'), model_seed=42, platform=PLATFORM))

Saved results/22_report_dependence.json
